# 通用模型数据生成（宿主机）

支持的模型：`mlp` (784→128→10) / `lenet5` (Conv+Pool+FC)

修改下方 `ARCH` 即可切换模型，其余流程完全一致。

In [ ]:
import sys
sys.path.insert(0, '.')
from model_zoo import *

## 1. 选择模型架构

In [ ]:
# ===== 在这里选择模型 =====
ARCH = 'lenet5'   # 'mlp' or 'lenet5'
# ==========================

SEED = 42
EPOCHS = {'mlp': 10, 'lenet5': 15}[ARCH]
NUM_TEST = 20
OUTPUT_DIR = f'{ARCH}_data'
MODEL_PATH = f'{ARCH}.pt'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Arch: {ARCH}, Device: {device}')

## 2. 训练

In [ ]:
model = build_model(ARCH)
float_acc = train(model, epochs=EPOCHS, device=device, seed=SEED)
torch.save(model.state_dict(), MODEL_PATH)
print(f'Saved to {MODEL_PATH}')

In [ ]:
# 或加载已有模型
# model = build_model(ARCH)
# model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
# model = model.to(device).eval()

## 3. 量化

In [ ]:
qparams = quantize(model.to(device).eval(), device)

## 4. INT8 准确率验证

In [ ]:
transform_raw = transforms.ToTensor()
test_set = datasets.MNIST('./data', train=False, download=True, transform=transform_raw)

correct_i8, correct_fp, total = 0, 0, len(test_set)
for i in range(total):
    img, label = test_set[i]
    img_u8 = np.clip(np.round(img.numpy().flatten() * 255), 0, 255).astype(np.uint8)
    pred_i8, _, _ = int8_infer(img_u8, qparams)
    if pred_i8 == label: correct_i8 += 1
    with torch.no_grad():
        pred_fp = model(img.unsqueeze(0).to(device)).argmax(1).item()
    if pred_fp == label: correct_fp += 1

print(f'Float32: {100.*correct_fp/total:.2f}% ({correct_fp}/{total})')
print(f'INT8:    {100.*correct_i8/total:.2f}% ({correct_i8}/{total})')
print(f'Drop:    {100.*(correct_fp-correct_i8)/total:.2f}%')

## 5. 导出 Hex

In [ ]:
test_images = []
test_labels = []
for i in range(NUM_TEST):
    img, label = test_set[i]
    test_images.append(np.clip(np.round(img.numpy().flatten()*255), 0, 255).astype(np.uint8))
    test_labels.append(int(label))

export_hex(qparams, OUTPUT_DIR, test_images, test_labels, NUM_TEST)

In [ ]:
import shutil
shutil.make_archive(OUTPUT_DIR, 'zip', '.', OUTPUT_DIR)
print(f'Created {OUTPUT_DIR}.zip — upload to PYNQ')